In [1]:
# -----------------------------
# Hugging Face auth (required if model is gated)
# -----------------------------
from huggingface_hub import login

# Path to your token file
token_path = "/data/liangz2/openi/hf_token.txt"

# Read first line (strip newline/whitespace)
with open(token_path, "r") as f:
    hf_token = f.readline().strip()

# Login using the token
login(token=hf_token, new_session=True)

print("✅ Hugging Face login successful.")


✅ Hugging Face login successful.


In [2]:
import os, json, zipfile, datetime, csv
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer

In [8]:
import transformers, torch

# -----------------------------
# Model ID
# -----------------------------
model_id = "aaditya/OpenBioLLM-Llama3-8B"

# -----------------------------
# Load pipeline (auto GPU if available)
# -----------------------------
pipe = transformers.pipeline(
    "text-generation",
    model=model_id,
    device_map="auto",              # automatically place on GPU(s)
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# -----------------------------
# Build plain prompt (NO chat template)
# -----------------------------
system_message = (
    "You are an expert and experienced healthcare and biomedical specialist "
    "with extensive medical knowledge and practical experience. "
    "Your name is OpenBioLLM, developed by Saama AI Labs. "
    "Provide explanations leveraging relevant anatomical structures, "
    "physiological processes, diagnostic criteria, and treatment guidelines. "
    "Use precise medical terminology while remaining clear and accessible."
)

user_message = "How can I split a 3mg or 4mg waefin pill to get a 2.5mg dose?"

prompt = f"""System: {system_message}

User: {user_message}

Assistant:"""

# -----------------------------
# Generate response
# -----------------------------
outputs = pipe(
    prompt,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
)

# -----------------------------
# Print only the generated completion
# -----------------------------
generated_text = outputs[0]["generated_text"]
response = generated_text[len(prompt):]

print(response.strip())



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


To split a 3mg or 4mg waefin pill to get a 2.5mg dose, you will need to use a pill splitter. A pill splitter is a small tool with two sharp edges that allows you to evenly divide a pill into two or more pieces. Make sure to wash your hands thoroughly before handling the pill and the pill splitter to maintain cleanliness and accuracy. Position the pill on the pill splitter and apply gentle pressure to cut it evenly into two pieces. Each piece should be approximately 2.5mg. It is important to note that pill splitting should only be done with certain types of pills and under the guidance of a healthcare professional. Always consult with your doctor or pharmacist before attempting to split any medication.


In [9]:
# -------------------------
# Paths & IDs
# -------------------------
MODEL_ID = "ContactDoctor/Bio-Medical-Llama-3-8B"
JSONL_PATH = "/data/liangz2/openi/harmony_set/openi_cxr_harmony_rl.jsonl"  # <-- change if needed

OUTPUT_DIR = "/data/liangz2/openi/qwen2_finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training length controls (tune for L4 vs A100)
MAX_LEN = 2048      # 1024–2048 for L4; 2048–4096 for A100 (if VRAM allows)
TRAIN_FRAC = 0.98   # train/eval split
SEED = 42

# QLoRA knobs
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

DEVICE_MAP = {"": 0}  # force onto GPU:0

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("eos:", tokenizer.eos_token, "pad:", tokenizer.pad_token)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ContactDoctor/Bio-Medical-Llama-3-8B.
403 Client Error. (Request ID: Root=1-6998ca62-4db217fa28daeccb2d4edf57;c18feab6-df4a-4167-9b99-626df06d7672)

Cannot access gated repo for url https://huggingface.co/ContactDoctor/Bio-Medical-Llama-3-8B/resolve/main/config.json.
Access to model ContactDoctor/Bio-Medical-Llama-3-8B is restricted and you are not in the authorized list. Visit https://huggingface.co/ContactDoctor/Bio-Medical-Llama-3-8B to ask for access.

In [4]:
#  Convert Harmony JSONL → Qwen SFT dataset
# Creates a Dataset with a single column text, formatted by Qwen’s chat template.

# Expected JSONL shapes (robust):

#  -- messages: list of {role, content}
#  -- harmony_prompt: list of {role, content}
#  -- fallback: prompt + completion


import json
from datasets import Dataset

def _extract_text_from_content(content):
    """
    Harmony messages may store content as:
      - string
      - list of parts: [{"type":"text","text":"..."}, ...]
      - dict with fields
    Return a plain string.
    """
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        out = []
        for part in content:
            if isinstance(part, str):
                out.append(part)
            elif isinstance(part, dict):
                # common: {"type":"text","text":"..."}
                if "text" in part and isinstance(part["text"], str):
                    out.append(part["text"])
                elif "content" in part and isinstance(part["content"], str):
                    out.append(part["content"])
                else:
                    out.append(str(part))
            else:
                out.append(str(part))
        return "\n".join([x for x in out if x.strip()])
    if isinstance(content, dict):
        # sometimes {"text": "..."}
        if "text" in content and isinstance(content["text"], str):
            return content["text"]
        if "content" in content and isinstance(content["content"], str):
            return content["content"]
        return json.dumps(content, ensure_ascii=False)
    return str(content)


def harmony_jsonl_to_qwen_text_dataset(
    jsonl_path: str,
    tokenizer,
    *,
    max_len: int = 2048,
    add_generation_prompt: bool = False,
    verbose: int = 1,
):
    """
    Convert Harmony JSONL to a HF Dataset with a single column: 'text'
    where 'text' is formatted using Qwen chat template.

    It will attempt to extract messages from common Harmony-like fields:
      - obj["messages"]  (list of {role, content})
      - obj["harmony_prompt"] (string or list or dict)
      - obj["prompt"] or obj["text"] as fallback

    Rows are skipped if no usable text/messages are found.
    """
    kept = []
    stats = {
        "total_lines": 0,
        "blank_lines": 0,
        "json_parse_fail": 0,
        "no_content": 0,
        "too_long_after_template": 0,
        "kept": 0,
    }

    def build_chat_messages(obj):
        # Preferred: "messages" list
        if "messages" in obj and isinstance(obj["messages"], list) and len(obj["messages"]) > 0:
            msgs = []
            for m in obj["messages"]:
                if not isinstance(m, dict):
                    continue
                role = m.get("role", None) or m.get("from", None)
                content = _extract_text_from_content(m.get("content", None))
                if role and content.strip():
                    # Qwen expects roles like system/user/assistant
                    # normalize common variants
                    role = role.lower()
                    if role in ("human", "prompt"):
                        role = "user"
                    if role in ("gpt", "bot", "model"):
                        role = "assistant"
                    msgs.append({"role": role, "content": content.strip()})
            return msgs if len(msgs) > 0 else None

        # Next: "harmony_prompt" (could be string or list)
        hp = obj.get("harmony_prompt", None)
        if hp is not None:
            txt = _extract_text_from_content(hp).strip()
            if txt:
                # If it’s already a flattened conversation, treat as user prompt
                return [{"role": "user", "content": txt}]

        # Fallback: "prompt" or "text"
        for k in ("prompt", "text"):
            if k in obj:
                txt = _extract_text_from_content(obj[k]).strip()
                if txt:
                    return [{"role": "user", "content": txt}]

        return None

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            stats["total_lines"] += 1
            line = line.strip()
            if not line:
                stats["blank_lines"] += 1
                continue

            try:
                obj = json.loads(line)
            except Exception:
                stats["json_parse_fail"] += 1
                continue

            msgs = build_chat_messages(obj)
            if not msgs:
                stats["no_content"] += 1
                continue

            # Apply Qwen chat template -> training text
            try:
                text = tokenizer.apply_chat_template(
                    msgs,
                    tokenize=False,
                    add_generation_prompt=add_generation_prompt,
                )
            except Exception:
                # If template fails, degrade gracefully to concatenation
                text = "\n\n".join([f"{m['role']}: {m['content']}" for m in msgs])

            # Optional length filter (approx by tokenizing)
            toks = tokenizer(
                text,
                truncation=False,
                add_special_tokens=False,
                return_attention_mask=False,
                return_token_type_ids=False,
            )["input_ids"]

            if len(toks) == 0:
                stats["no_content"] += 1
                continue

            if len(toks) > max_len:
                # If you prefer truncation instead of dropping, change this behavior:
                # text = tokenizer.decode(toks[:max_len], skip_special_tokens=False)
                stats["too_long_after_template"] += 1
                continue

            kept.append({"text": text})
            stats["kept"] += 1

    if verbose:
        print("Converter stats:", stats)
        if stats["kept"] == 0:
            print("\n⚠️ No rows kept. Common reasons:")
            print(" - jsonl_path points to the wrong file or file is empty")
            print(" - the JSONL schema doesn't contain messages/text fields the converter expects")
            print(" - every example exceeds max_len and gets dropped")
            print("\nNext debug step: print a few raw lines from the JSONL.")

    return Dataset.from_list(kept)


# ---- Run conversion ----
JSONL_PATH = "/data/liangz2/openi/harmony_set/openi_cxr_harmony_rl.jsonl"  # adjust if needed
raw_ds = harmony_jsonl_to_qwen_text_dataset(JSONL_PATH, tokenizer, max_len=MAX_LEN, verbose=1)

print(raw_ds)
if len(raw_ds) > 0:
    print("Preview:\n", raw_ds[0]["text"][:800])
else:
    # Debug: show first 3 raw lines to see schema
    print("\n--- Debug: first 3 non-empty lines of JSONL ---")
    shown = 0
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                print(line[:500])
                shown += 1
                if shown >= 3:
                    break

Converter stats: {'total_lines': 7902, 'blank_lines': 0, 'json_parse_fail': 0, 'no_content': 0, 'too_long_after_template': 0, 'kept': 7902}
Dataset({
    features: ['text'],
    num_rows: 7902
})
Preview:
 <|im_start|>system
You are a radiology assistant. Decide whether the STATEMENT is supported by the EVIDENCE. Output only TRUE or FALSE.<|im_end|>
<|im_start|>user
EVIDENCE
normal: no
labels_13: ['lung opacity', 'lung lesion', 'atelectasis', 'consolidation']
mesh_major: ['Opacity/lung/upper lobe/right', 'Pulmonary Atelectasis/upper lobe/right', 'Opacity/lung/lingula']
mesh_automatic: ['atelectases', 'mass lesion', 'opacity', 'Atelectasis', 'Ribs']
summary: Chest X-ray shows right upper lobe lung opacity suspicious for lung lesion with associated atelectasis or consolidation, and a left midlung opacity; heart size normal, no pleural effusion or pneumothorax.
caption: The chest X-ray reveals increased opacity in the right upper lobe, which may indicate a mass lesion accompanied by atele

In [5]:
# Train / Eval split
raw_ds = raw_ds.shuffle(seed=SEED)
n = len(raw_ds)
n_train = int(n * TRAIN_FRAC)

train_ds = raw_ds.select(range(n_train))
eval_ds = raw_ds.select(range(n_train, n)) if n_train < n else None

ds = DatasetDict({"train": train_ds})
if eval_ds is not None and len(eval_ds) > 0:
    ds["eval"] = eval_ds

print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 7743
    })
    eval: Dataset({
        features: ['text'],
        num_rows: 159
    })
})


In [6]:
def tokenize_sft(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

tokenized_train = ds["train"].map(tokenize_sft, batched=True, remove_columns=ds["train"].column_names)

tokenized_eval = None
if "eval" in ds and len(ds["eval"]) > 0:
    tokenized_eval = ds["eval"].map(tokenize_sft, batched=True, remove_columns=ds["eval"].column_names)

print(tokenized_train)

Map:   0%|          | 0/7743 [00:00<?, ? examples/s]

Map:   0%|          | 0/159 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 7743
})


In [7]:
# ! pip install -U pip setuptools wheel
# ! pip install -U bitsandbytes

In [8]:
from transformers import BitsAndBytesConfig, DataCollatorForLanguageModeling

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map=DEVICE_MAP,
    torch_dtype="auto",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

# Qwen2 common target modules; adjust if your Transformers build differs.
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 73,859,072 || all params: 1,617,573,376 || trainable%: 4.5660


In [9]:
# -----------------------------
# set training arguments
# -----------------------------
from transformers import TrainingArguments

do_eval = tokenized_eval is not None

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=10,
    save_steps=200,
    eval_steps=200 if do_eval else None,
    eval_strategy="steps" if do_eval else "no",   # ✅ renamed
    save_strategy="steps",
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8,
    gradient_checkpointing=True,
    report_to=[],
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    seed=SEED,
)
print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=200,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,
f

In [10]:
def save_lora_adapter(trainer, output_dir, adapter_name="lora"):
    """
    Save LoRA adapter weights and tokenizer.
    
    Args:
        trainer: SFTTrainer or Trainer instance
        output_dir (str): base output directory
        adapter_name (str): subfolder name for LoRA adapter
    """
    save_path = os.path.join(output_dir, adapter_name)
    os.makedirs(save_path, exist_ok=True)

    model = trainer.model

    # Save LoRA adapter weights
    model.save_pretrained(save_path)

    # Save tokenizer (important for inference)
    if trainer.tokenizer is not None:
        trainer.tokenizer.save_pretrained(save_path)

    # Optional: save training state
    trainer.state.save_to_json(os.path.join(save_path, "trainer_state.json"))

    print(f"✅ LoRA adapter saved to: {save_path}")

In [11]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    processing_class=tokenizer,  # older TRL compatibility
)

train_result = trainer.train()
print(train_result)
save_lora_adapter(trainer, OUTPUT_DIR)

METRICS_CSV = os.path.join(OUTPUT_DIR, "training_metrics.csv")
log_history = trainer.state.log_history

if len(log_history) == 0:
    print("⚠️ No metrics found in trainer.state.log_history")
else:
    fieldnames = sorted({k for row in log_history for k in row.keys()})
    with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in log_history:
            writer.writerow(row)
    print("✅ Saved metrics:", METRICS_CSV)

Truncating train dataset:   0%|          | 0/7743 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/159 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` 

Step,Training Loss,Validation Loss
200,0.582500,0.486422
400,0.541000,0.407327
600,0.466400,0.369489
800,0.428900,0.343409
1000,0.366500,0.324984
1200,0.361200,0.311311
1400,0.328200,0.296885
1600,0.326000,0.283557
1800,0.282000,0.249991
2000,0.218700,0.243333


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=9680, training_loss=0.14705515823149976, metrics={'train_runtime': 30547.569, 'train_samples_per_second': 2.535, 'train_steps_per_second': 0.317, 'total_flos': 1.87070849412864e+17, 'train_loss': 0.14705515823149976})


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


✅ LoRA adapter saved to: /data/liangz2/openi/qwen2_finetuned/lora
✅ Saved metrics: /data/liangz2/openi/qwen2_finetuned/training_metrics.csv
